# Field Compounding Validation — Colab GPU Run

Reproduces Module 12 benchmarks (sections 3–14), compound field experiment (section 15), and `validate_results.py`.

**Repos:** `field-compounding-validation` (integration) + `visual-robot-learning` (Module 11 coupling / optional `MODULE11_RESULTS`).

Runtime: use **GPU** runtime (T4/A100) for torch-backed trials; CPU works with `--fast` for smoke only.


In [ ]:
# @title Runtime check
import sys
print("Python", sys.version)
try:
    import torch
    print("torch", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")


In [ ]:
# @title Clone repositories
import os
from pathlib import Path

WORK = Path("/content")
os.chdir(WORK)

FIELD_REPO = "https://github.com/thatrandomasiandev/field-compounding-validation.git"
M11_REPO = "https://github.com/thatrandomasiandev/visual-robot-learning.git"
FIELD_BRANCH = "integration"

!rm -rf field-compounding-validation visual-robot-learning
!git clone --branch integration --single-branch https://github.com/thatrandomasiandev/field-compounding-validation.git field-compounding-validation
!git clone --depth 1 https://github.com/thatrandomasiandev/visual-robot-learning.git visual-robot-learning

FIELD_ROOT = WORK / "field-compounding-validation"
M11_ROOT = WORK / "visual-robot-learning"
print("Field repo:", FIELD_ROOT)
print("Module 11 repo:", M11_ROOT)


In [ ]:
# @title Install packages (field_compounding + cv_robotics)
import os
import sys
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
M11_ROOT = Path("/content/visual-robot-learning")
os.chdir(FIELD_ROOT)
sys.path.insert(0, str(FIELD_ROOT / "src"))

!pip install -q -e ".[dev]"
!pip install -q -e {M11_ROOT}

import cv_robotics
print("cv_robotics:", cv_robotics.__file__)

os.environ.setdefault("FIELD_COMPOUNDING_DEVICE", "cuda")

m11_results = M11_ROOT / "results"
if m11_results.is_dir():
    os.environ["MODULE11_RESULTS"] = str(m11_results)
    print("MODULE11_RESULTS =", os.environ["MODULE11_RESULTS"])
else:
    print("Using bundled observatory/module11_coupling.json")


In [ ]:
# @title Configuration
import os
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
os.chdir(FIELD_ROOT)

# Set FIELD_COMPOUNDING_FAST=1 for smoke; 0 for full Colab GPU runs
N_TRIALS = 20  # @param {type:"integer"}
FAST = False  # @param {type:"boolean"}

if FAST:
    os.environ["FIELD_COMPOUNDING_FAST"] = "1"
else:
    os.environ.pop("FIELD_COMPOUNDING_FAST", None)

FAST_FLAG = " --fast" if FAST or os.environ.get("FIELD_COMPOUNDING_FAST") == "1" else ""
print(f"n_trials={N_TRIALS}, fast={FAST or os.environ.get('FIELD_COMPOUNDING_FAST') == '1'}")


In [ ]:
# @title Run benchmarks modules 03–14
import os
import subprocess
import sys
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
os.chdir(FIELD_ROOT)

cmd = [sys.executable, "scripts/run_benchmark.py", "--module", "all", "--n-trials", str(N_TRIALS)]
if FAST or os.environ.get("FIELD_COMPOUNDING_FAST") == "1":
    cmd.append("--fast")
print(" ".join(cmd))
proc = subprocess.run(cmd)
if proc.returncode != 0:
    raise RuntimeError(f"Benchmark failed with exit code {proc.returncode}. Scroll up for traceback.")


In [ ]:
# @title Compound field experiment (section 15)
import os
import subprocess
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
os.chdir(FIELD_ROOT)

cmd = ["python", "scripts/run_compound_field_experiment.py", "--n-trials", str(N_TRIALS)]
if FAST or os.environ.get("FIELD_COMPOUNDING_FAST") == "1":
    cmd.append("--fast")
subprocess.run(cmd, check=True)


In [ ]:
# @title Validate results
import subprocess
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
os.chdir(FIELD_ROOT)

min_trials = 5 if (FAST or __import__("os").environ.get("FIELD_COMPOUNDING_FAST") == "1") else 20
subprocess.run(
    ["python", "scripts/validate_results.py", "--min-trials", str(min_trials)],
    check=True,
)


In [ ]:
# @title Optional: pytest (integration sanity)
import subprocess
from pathlib import Path

FIELD_ROOT = Path("/content/field-compounding-validation")
subprocess.run(["pytest", "-q"], cwd=FIELD_ROOT, check=False)


In [ ]:
# @title List result artifacts
from pathlib import Path
import json

results = Path("/content/field-compounding-validation/results")
for p in sorted(results.glob("section_*.json")):
    data = json.loads(p.read_text())
    if p.stem == "section_15":
        print(p.name, "conditions:", len(data.get("conditions", {})))
    else:
        print(p.name, "trials:", data.get("n_successful"), "/", data.get("n_trials"))
